# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page.
i will analyze content performance signals from the available search data.(trailing 90-day performance data.)

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("/content/content_refresh_anonymized.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
- clicks_90d
- impressions_90d
- ctr
- avg_position
- word_count

Label:
- trend_direction (used as an outcome)

Context:
- content_type
- client_id

Excluded:
- content_id because it is only an identifier.
- trend_pct because it creates the label.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df[['clicks_90d','impressions_90d','ctr','avg_position','word_count']].head()

,clicks_90d,impressions_90d,ctr,avg_position,word_count
0,29,3803,0.76,10.6,3221.0
1,7,15320,0.05,20.3,2481.0
2,11,12581,0.09,36.5,3515.0
3,58,11751,0.49,6.2,NaN
4,24,19140,0.13,44.0,2803.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will check the dataset size and missing values.I selected five features that are available from existing content and search performance data before making decisions

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE thisprint("Total rows:", len(df))
print("Total content pages:", df.shape[0])
print("Duplicate content IDs:", df['content_id'].duplicated().sum())
print("\nMissing values:")
print(df[['impressions_90d','clicks_90d','ctr','avg_position']].isnull().sum())
print("\nPerformance columns checked:")
print(['impressions_90d','clicks_90d','sessions_90d'])

feature_df = df[['clicks_90d',
                 'impressions_90d',
                 'ctr',
                 'avg_position',
                 'word_count']]

feature_df.head()

Total content pages: 30000
Duplicate content IDs: 0

Missing values:
impressions_90d    0
clicks_90d         0
ctr                0
avg_position       0
dtype: int64

Performance columns checked:
['impressions_90d', 'clicks_90d', 'sessions_90d']


,clicks_90d,impressions_90d,ctr,avg_position,word_count
0,29,3803,0.76,10.6,3221.0
1,7,15320,0.05,20.3,2481.0
2,11,12581,0.09,36.5,3515.0
3,58,11751,0.49,6.2,NaN
4,24,19140,0.13,44.0,2803.0


Feature availability:

- clicks_90d: available from historical search performance data.
- impressions_90d: available from historical visibility data.
- ctr: calculated from clicks and impressions.
- avg_position: available from search ranking data.
- word_count: available from content information.

I tested a label-derived feature to understand data leakage. Since this feature comes directly from the label, it should not be used for modeling.

In [32]:
df['leak_feature'] = df['trend_direction']
df[['trend_direction', 'leak_feature']].head()
df = df.drop(columns=['leak_feature'])

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data shows relationships between search signals and performance.
It cannot prove that a signal causes ranking changes or predict Google's algorithm.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Dataset limitation checked.")

Dataset limitation checked.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.